In [1]:
import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("fitcheck")
    print("HF token loaded")
except Exception as e:
    print("No HF token:", e)

HF token loaded


In [2]:
!rm -rf /kaggle/working/fitcheck
!git clone -q https://github.com/Anassbzdd/fitcheck.git /kaggle/working/fitcheck
!cd /kaggle/working/fitcheck && pip install -q -e . && pip install -q -r scripts/requirements-measure.txt
!ls /kaggle/working/fitcheck/scripts/

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fitcheck-llm (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.5 MB/s eta 0:00:00:00:01
measure_infer.py  measure.py  requirements-measure.txt


In [3]:
!python -c "import torch, peft, transformers, torchao; print('torch:', torch.__version__); print('peft:', peft.__version__); print('transformers:', transformers.__version__); print('torchao:', torchao.__version__)"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
torch: 2.10.0+cu128
peft: 0.19.1
transformers: 5.0.0
torchao: 0.18.0


In [4]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-135M --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/E1a.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1105.72it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-135M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 138,201,408 logical | 3,686,400 trainable
          base 134,515,008 vs fitcheck P 134,515,008  (+0)  [OK]
  optimizer states: 28 MiB observed, dtype float32
  step time: 0.15s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    all

In [5]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-135M --quant none --precision fp16 --lora-r 32 --batch-size 4 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/E1b.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1119.92it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-135M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=4, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 138,201,408 logical | 3,686,400 trainable
          base 134,515,008 vs fitcheck P 134,515,008  (+0)  [OK]
  optimizer states: 28 MiB observed, dtype float32
  step time: 0.20s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    all

In [6]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-135M --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 2048 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/E1c.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1116.92it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-135M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=2048, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 138,201,408 logical | 3,686,400 trainable
          base 134,515,008 vs fitcheck P 134,515,008  (+0)  [OK]
  optimizer states: 28 MiB observed, dtype float32
  step time: 0.24s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    al

In [7]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-1.7B --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/E2a.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 218/218 [00:01<00:00, 127.76it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-1.7B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,723,959,296 logical | 12,582,912 trainable
          base 1,711,376,384 vs fitcheck P 1,711,376,384  (+0)  [OK]
  optimizer states: 96 MiB observed, dtype float32
  step time: 0.25s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
 

In [8]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-1.7B --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 512 --gpu t4 2>&1 | tee /kaggle/working/E2b.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 218/218 [00:01<00:00, 134.59it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-1.7B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=512, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,723,959,296 logical | 12,582,912 trainable
          base 1,711,376,384 vs fitcheck P 1,711,376,384  (+0)  [OK]
  optimizer states: 96 MiB observed, dtype float32
  step time: 0.31s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    3,312 MiB
    resident before first step              3,312 MiB
    gradients (after backward)                 48 MiB
    peak allocated  

In [9]:
!cd /kaggle/working/fitcheck && python scripts/measure.py Qwen/Qwen2.5-0.5B --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/E3a.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 630.61it/s, Materializing param=model.norm.weight]                              

  Qwen/Qwen2.5-0.5B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 498,358,144 logical | 4,325,376 trainable
          base 494,032,768 vs fitcheck P 494,032,768  (+0)  [OK]
  optimizer states: 33 MiB observed, dtype float32
  step time: 0.20s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated aft

In [10]:
!cd /kaggle/working/fitcheck && python scripts/measure.py Qwen/Qwen2.5-0.5B --quant none --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512 --gpu t4 2>&1 | tee /kaggle/working/E3b.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 577.16it/s, Materializing param=model.norm.weight]                              model.layers.2.mlp.down_proj.weight]

  Qwen/Qwen2.5-0.5B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 498,358,144 logical | 4,325,376 trainable
          base 494,032,768 vs fitcheck P 494,032,768  (+0)  [OK]
  optimizer states: 33 MiB observed, dtype float32
  step time: 0.24s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      967 MiB
    resident before first step                967 MiB
    gradients (after backward)                 16 MiB


In [11]:
!cd /kaggle/working/fitcheck && python scripts/measure.py Qwen/Qwen2.5-1.5B --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/E4a.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 241.62it/s, Materializing param=model.norm.weight]                              

  Qwen/Qwen2.5-1.5B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,552,430,592 logical | 8,716,288 trainable
          base 1,543,714,304 vs fitcheck P 1,543,714,304  (+0)  [OK]
  optimizer states: 66 MiB observed, dtype float32
  step time: 0.23s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocat

In [12]:
!cd /kaggle/working/fitcheck && python scripts/measure.py Qwen/Qwen2.5-1.5B --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 512 --gpu t4 2>&1 | tee /kaggle/working/E4b.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 241.67it/s, Materializing param=model.norm.weight]                              

  Qwen/Qwen2.5-1.5B  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=512, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,552,430,592 logical | 8,716,288 trainable
          base 1,543,714,304 vs fitcheck P 1,543,714,304  (+0)  [OK]
  optimizer states: 66 MiB observed, dtype float32
  step time: 0.22s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    2,979 MiB
    resident before first step              2,979 MiB
    gradients (after backward)                 33 MiB
    peak allocated  (tensor by

In [13]:
!cd /kaggle/working/fitcheck && python scripts/measure.py unsloth/gemma-2-2b --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 512 --gpu t4 --flash-attn --attn-impl sdpa 2>&1 | tee /kaggle/working/E5a.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
Loading weights: 100%|██████████| 288/288 [00:02<00:00, 104.21it/s, Materializing param=model.norm.weight]                                

  unsloth/gemma-2-2b  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=512, fp16, adamw, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 2,627,121,408 logical | 12,779,520 trainable
          base 2,614,341,888 vs fitcheck P 2,614,341,888  (+0)  [OK]
  optimizer states: 98 MiB observed, dtype float32
  step time: 0.43s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    all

In [14]:
!cd /kaggle/working/fitcheck && python scripts/measure.py unsloth/gemma-2-2b --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 512 --gpu t4 2>&1 | tee /kaggle/working/E5b.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 288/288 [00:02<00:00, 119.64it/s, Materializing param=model.norm.weight]                                       | 25/288 [00:01<04:05,  1.07it/s, Materializing param=model.layers.2.mlp.down_proj.weight]  

  unsloth/gemma-2-2b  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=512, fp16, adamw, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 2,627,121,408 logical | 12,779,520 trainable
          base 2,614,341,888 vs fitcheck P 2,614,341,888  (+0)  [OK]
  optimizer states: 98 MiB observed, dtype float32
  step time: 0.43s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    5,036 MiB
    resident before first step     

In [15]:
!cd /kaggle/working/fitcheck && python scripts/measure.py JackFram/llama-160m --no-lora --quant none --precision fp16 --batch-size 2 --seq-len 1024 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/D1_fullft.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 111/111 [00:00<00:00, 776.07it/s, Materializing param=model.norm.weight]                              

  JackFram/llama-160m  on  Tesla T4
  full FT, bs=2, seq=1024, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 162,417,408 logical | 162,417,408 trainable
          base 162,417,408 vs fitcheck P 162,417,792  (-384)  [OK]
  optimizer states: 1,239 MiB observed, dtype float32
  step time: 0.89s

  MEASURED
    CUDA context (at peak)                    139 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      319 MiB
    resident before first step                621 MiB
    gradients (after backward)                620 MiB
    peak allocated  (tensor byte

In [16]:
!cd /kaggle/working/fitcheck && python scripts/measure.py HuggingFaceTB/SmolLM2-360M --quant none --precision fp16 --lora-r 32 --batch-size 1 --seq-len 4096 --grad-checkpoint --gpu t4 2>&1 | tee /kaggle/working/D3_seq4096.txt


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 712.41it/s, Materializing param=model.norm.weight]                              

  HuggingFaceTB/SmolLM2-360M  on  Tesla T4
  LoRA r=32 [q,k,v,o], bs=1, seq=4096, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 368,374,720 logical | 6,553,600 trainable
          base 361,821,120 vs fitcheck P 361,821,120  (+0)  [OK]
  optimizer states: 50 MiB observed, dtype float32
  step time: 4.50s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                      729 MiB
    resident before first step                729 MiB
    gradients (after backward)                 25 MiB
    peak allocated  